# OpenCV: from pixels to object measurements

A generated image contains objects that need to be counted and measured. Start with the image representation, form a foreground mask, remove selected artifacts and measure connected regions. At every step, compare the transformed image with the original so that a cleaner-looking mask is not mistaken for a more accurate one.

OpenCV provides image-processing and computer-vision algorithms. We will create three colored shapes, separate them from a background, inspect their edges, and measure their boundaries.

The scene is generated teaching data. No camera, downloaded images, or trained models are required. Run cells in order. Measurements are in pixels, not physical units; real-world measurements require calibration.

## 1. Install and import

If needed, run `%pip install opencv-python-headless numpy matplotlib` in a separate cell. The headless distribution supports these examples without desktop window functions. Import it as `cv2`; use only one OpenCV distribution in an environment.

We display images with Matplotlib. OpenCV color arrays commonly use BGR channel order, while Matplotlib expects RGB.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv

print("OpenCV:", cv.__version__)
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)

def show_image(image, title):
    fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
    if image.ndim == 2:
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
    plt.show()

## 2. Generate a scene

NumPy arrays use `(height, width, channels)`, while drawing coordinates are `(x, y)`. Pixel values are 8-bit integers from 0 to 255.

We draw a red rectangle, a green circle, and a blue triangle on a light background. Filled drawing operations modify the target array.

In [ ]:
scene = np.full((320, 600, 3), 240, dtype=np.uint8)
cv.rectangle(scene, (40, 80), (160, 240), (40, 40, 210), thickness=-1)
cv.circle(scene, (300, 160), 70, (40, 170, 40), thickness=-1)
triangle = np.array([[440, 240], [520, 70], [570, 240]], dtype=np.int32)
cv.fillPoly(scene, [triangle], (210, 70, 40))
show_image(scene, "Three generated objects")
print("Shape:", scene.shape, "dtype:", scene.dtype)

## 3. Save and load pixels

`imwrite` returns a success flag; check it. `imread` can return `None` when loading fails, so check before using the result. This PNG stores the test image losslessly.

`IMREAD_COLOR` loads three color channels. Other flags can preserve an alpha channel or load grayscale, changing the returned shape.

In [ ]:
source_path = output_dir / "opencv_source.png"
if not cv.imwrite(str(source_path), scene):
    raise OSError("Could not save the source image")
loaded = cv.imread(str(source_path), cv.IMREAD_COLOR)
if loaded is None:
    raise OSError("Could not load the source image")
print("Loaded shape:", loaded.shape)
print("Lossless round-trip:", np.array_equal(scene, loaded))

## 4. Keep colors consistent while changing representations

Predict how the scene should look before conversion. Inspect a known color after switching between BGR and RGB. A swapped channel order can look plausible while breaking color-based segmentation.

Grayscale combines color information into one intensity channel. HSV separates hue, saturation, and value, which can help isolate strongly colored objects.

For OpenCV's standard 8-bit HSV conversion, hue ranges from 0 to 179, while saturation and value range from 0 to 255. These ranges differ from floating-point HSV conventions.

In [ ]:
gray = cv.cvtColor(loaded, cv.COLOR_BGR2GRAY)
hsv = cv.cvtColor(loaded, cv.COLOR_BGR2HSV)
print("Rectangle BGR:", loaded[100, 100].tolist())
print("Rectangle HSV:", hsv[100, 100].tolist())
show_image(gray, "Grayscale intensities")

## 5. Crop and resize

Array slicing selects `[top:bottom, left:right]`, excluding the final boundary. A slice can share memory with its source; `.copy()` makes an independent region.

`resize` takes `(width, height)`. INTER_AREA is useful for shrinking images. Resize binary masks with nearest-neighbor interpolation to retain discrete labels.

In [ ]:
region = loaded[60:260, 20:180].copy()
small = cv.resize(loaded, (300, 160), interpolation=cv.INTER_AREA)
print("Region shape:", region.shape, "Small image shape:", small.shape)
show_image(region, "Rectangle region")

## 6. Smooth generated noise

Create reproducible Gaussian noise, add it using a wider numeric type, and clip before converting to uint8. Direct arithmetic on uint8 arrays can wrap around.

Gaussian blur reduces local variation but also softens boundaries. Its kernel dimensions must be positive and odd for this call.

In [ ]:
rng = np.random.default_rng(42)
noise = rng.normal(0, 12, gray.shape)
noisy = np.clip(gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)
blurred = cv.GaussianBlur(noisy, (5, 5), sigmaX=1.2)
show_image(blurred, "Noisy grayscale image after smoothing")

## 7. Decide which pixels belong to an object

Before thresholding, identify foreground and background in the source. Compare the resulting mask with that expectation. A threshold should be justified by separation in the image, not by the convenience of a familiar constant.

Our shapes are darker than the background. Inverse binary thresholding makes dark objects white and the background black. Otsu's method chooses a global threshold from the intensity histogram.

This controlled scene has strong contrast. Uneven illumination, shadows, and overlapping intensities can make global thresholding inadequate.

In [ ]:
otsu_value, otsu_mask = cv.threshold(blurred, 0, 255, cv.THRESH_BINARY_INV | cv.THRESH_OTSU)
print("Otsu threshold:", otsu_value)
show_image(otsu_mask, "Objects selected by grayscale thresholding")

## 8. Segment by saturation

The objects are colorful while the background is nearly gray. Select saturated pixels with `inRange` across the full hue range. The result is a single-channel uint8 mask.

This rule is designed for this scene. It is not a general object detector and will also select any other sufficiently saturated region.

In [ ]:
color_mask = cv.inRange(hsv, np.array([0, 80, 0], dtype=np.uint8), np.array([179, 255, 255], dtype=np.uint8))
selected = cv.bitwise_and(loaded, loaded, mask=color_mask)
show_image(color_mask, "Saturated foreground mask")
show_image(selected, "Selected color pixels")

## 9. Remove noise without erasing small objects

Predict whether the morphology operation will remove small regions or fill small gaps. Compare object counts and boundaries before and after. A kernel that removes noise can also remove a genuine object of similar size.

Opening erodes then dilates, helping remove isolated foreground specks. Closing dilates then erodes, helping fill small gaps. Kernel size determines which structures are affected.

We deliberately add a small speck and a small hole. Morphology can also change genuine boundaries or remove small real objects, so choose its scale deliberately.

In [ ]:
dirty_mask = color_mask.copy()
cv.circle(dirty_mask, (15, 15), 1, 255, -1)
cv.circle(dirty_mask, (100, 150), 1, 0, -1)
kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
opened = cv.morphologyEx(dirty_mask, cv.MORPH_OPEN, kernel)
clean_mask = cv.morphologyEx(opened, cv.MORPH_CLOSE, kernel)
show_image(clean_mask, "Mask after opening and closing")

## 10. Detect edges

Canny detects intensity boundaries using two thresholds: strong edges survive, and weaker edges can survive when connected to strong ones. It produces thin boundaries rather than filled object regions.

Edges and masks answer different questions. We use the filled mask for area measurement, not the Canny output.

In [ ]:
edges = cv.Canny(cv.GaussianBlur(gray, (5, 5), 1), threshold1=50, threshold2=150)
show_image(edges, "Canny boundaries")

## 11. Distinguish image measurements from physical measurements

Check whether the measured regions correspond to the visible objects. Explain the units of area and perimeter. Pixel area is not physical area without a scale calibration, and contour area need not equal the count of foreground pixels.

`RETR_EXTERNAL` keeps only outer boundaries; it ignores holes and nested contours. `CHAIN_APPROX_SIMPLE` compresses redundant boundary points. Sort explicitly because contour order has no semantic meaning.

Contour area is geometric area enclosed by a boundary and can differ from the number of white pixels. Moments give a centroid when the contour has nonzero area. Bounding rectangles use `(x, y, width, height)`.

In [ ]:
contours, hierarchy = cv.findContours(clean_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
object_contours = sorted([contour for contour in contours if cv.contourArea(contour) >= 500],
                         key=lambda contour: cv.boundingRect(contour)[0])
measurements = []
for object_id, contour in enumerate(object_contours, start=1):
    x, y, width, height = cv.boundingRect(contour)
    moments = cv.moments(contour)
    if moments["m00"] == 0:
        continue
    measurements.append({"object_id": object_id, "x": x, "y": y, "width": width, "height": height,
        "contour_area_px2": float(cv.contourArea(contour)),
        "perimeter_px": float(cv.arcLength(contour, True)),
        "center_x": moments["m10"] / moments["m00"],
        "center_y": moments["m01"] / moments["m00"]})
print(json.dumps(measurements, indent=2))

## 12. Label connected components

Connected components group adjacent foreground pixels. Label 0 is background, so subtract one from the returned label count to count foreground groups.

The component area counts pixels. Comparing it with contour area illustrates why the two measures need not match exactly.

In [ ]:
component_count, component_labels, component_stats, component_centroids = cv.connectedComponentsWithStats(clean_mask, connectivity=8)
component_areas = component_stats[1:, cv.CC_STAT_AREA]
print("Foreground components:", component_count - 1)
print("Component pixel areas:", component_areas.tolist())
print("Total foreground pixels:", cv.countNonZero(clean_mask))

## 13. Approximate boundaries with polygons

`approxPolyDP` simplifies a contour using a distance tolerance. Here the tolerance is 2% of its perimeter. Fewer vertices make broad geometric structure visible.

The vertex count depends on the mask and tolerance. Three or four vertices in this scene are descriptive clues, not a reliable classifier for arbitrary photographs.

In [ ]:
polygon_overlay = loaded.copy()
vertex_counts = []
for contour in object_contours:
    perimeter = cv.arcLength(contour, True)
    polygon = cv.approxPolyDP(contour, 0.02 * perimeter, True)
    vertex_counts.append(len(polygon))
    cv.polylines(polygon_overlay, [polygon], True, (0, 0, 0), 2)
print("Polygon vertex counts from left to right:", vertex_counts)
show_image(polygon_overlay, "Simplified boundaries")

## 14. Rotate with an affine transform

An affine matrix maps coordinates through transformations such as rotation, scaling, and translation. `warpAffine` takes an explicit output size.

This output keeps the original canvas dimensions, so content near its edges could be clipped. Nearest-neighbor interpolation is appropriate when transforming a label mask; the color image here uses linear interpolation.

In [ ]:
height, width = loaded.shape[:2]
rotation_matrix = cv.getRotationMatrix2D((width / 2, height / 2), angle=12, scale=0.85)
rotated = cv.warpAffine(loaded, rotation_matrix, (width, height), flags=cv.INTER_LINEAR,
                        borderMode=cv.BORDER_CONSTANT, borderValue=(240, 240, 240))
show_image(rotated, "Affine rotation and scale")

## 15. Complete exercise: export an object report

Annotate each retained contour with its ID, bounding rectangle, and centroid. Export the mask and annotated image as PNG, a comparison figure, and measurements as JSON and CSV.

The source is synthetic and separated into three objects. Touching objects, low contrast, holes, or a different background would require revisiting the segmentation choices. Running this cell replaces these named files in `outputs`.

In [ ]:
annotated = loaded.copy()
for contour, record in zip(object_contours, measurements):
    x, y, width, height = [record[key] for key in ["x", "y", "width", "height"]]
    cv.drawContours(annotated, [contour], -1, (0, 0, 0), 2)
    cv.rectangle(annotated, (x, y), (x + width - 1, y + height - 1), (80, 80, 80), 1)
    center = (round(record["center_x"]), round(record["center_y"]))
    cv.circle(annotated, center, 4, (255, 255, 255), -1)
    cv.putText(annotated, f"Object {record['object_id']}", (x, max(20, y - 12)),
               cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv.LINE_AA)
for filename, image in [("opencv_mask.png", clean_mask), ("opencv_annotated.png", annotated)]:
    if not cv.imwrite(str(output_dir / filename), image):
        raise OSError(f"Could not save {filename}")
(output_dir / "opencv_measurements.json").write_text(json.dumps(measurements, indent=2) + "\n", encoding="utf-8")
fields = ["object_id", "x", "y", "width", "height", "contour_area_px2", "perimeter_px", "center_x", "center_y"]
with (output_dir / "opencv_measurements.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(measurements)
report, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
for ax, image, title in zip(axes, [loaded, clean_mask, annotated], ["Source", "Foreground mask", "Measured objects"]):
    if image.ndim == 2:
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
report.savefig(output_dir / "opencv_object_report.png", dpi=150)
plt.show()
print(f"Exported {len(measurements)} object measurements and PNG images.")

## 16. Practice and troubleshooting

1. Add a fourth separated shape and check the component and contour counts.
2. Move two shapes until they touch. Explain why the mask may yield one connected component.
3. Increase the morphology kernel size and inspect which details disappear.
4. Compare contour area with component pixel counts for each shape.

| Symptom | What to check |
| --- | --- |
| Red and blue are swapped | Convert BGR to RGB before Matplotlib display |
| `imread` gives None | Verify the path, file format, and read permissions |
| Cropping selects the wrong region | NumPy indexes y then x |
| Background is detected as an object | Check foreground polarity in the binary mask |
| Small objects disappear | Inspect smoothing, morphology, and minimum-area filtering |
| Several objects become one | Check whether their foreground pixels touch |
| Pixel measurements are mistaken for centimeters | Calibrate scale and account for perspective first |
| GUI windows do not work | Use notebook plots with the headless package |

See the official [image-processing tutorials](https://docs.opencv.org/4.x/d2/d96/tutorial_py_table_of_contents_imgproc.html) and [contour introduction](https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html).

## Explain what the evidence supports

Choose a counted region and explain how its pixels became a mask and then a measurement. Identify one lighting or overlap change that would challenge the method and a diagnostic image you would inspect.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: count objects under uneven illumination and noise

A fixed global threshold works on a clean synthetic scene but fails when one side is darker. Generate an unevenly lit grayscale scene with four known objects, add noise, then compare a global threshold with illumination normalization. Measure mask overlap with the known fixture and inspect connected components. This simulates a camera-quality challenge; it is not a calibrated industrial inspection system.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the mask error before counting

A foreground mask should describe object pixels, not all bright pixels. Under a lighting gradient, background can be brighter than an object elsewhere. A single threshold therefore has no guarantee of separating both. Count alone is insufficient: one spurious region and one missed object can cancel out numerically. Compare pixel overlap as well as component count.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

import cv2 as cv
import matplotlib.pyplot as plt
p2_rng=np.random.default_rng(4)
p2_truth=np.zeros((240,360),dtype=np.uint8)
for p2_center in [(60,60),(170,65),(280,70),(180,175)]:cv.circle(p2_truth,p2_center,20,255,-1)
p2_light=np.tile(np.linspace(35,155,360),(240,1))
p2_scene=np.clip(p2_light+(p2_truth>0)*65+p2_rng.normal(0,4,p2_truth.shape),0,255).astype('uint8')
_,p2_global=cv.threshold(p2_scene,145,255,cv.THRESH_BINARY)
# Wide smoothing approximates illumination; cast to float before subtraction to avoid uint8 wraparound.
p2_background=cv.GaussianBlur(p2_scene.astype('float32'),(0,0),25)
p2_residual=p2_scene.astype('float32')-p2_background
p2_corrected=(p2_residual>25).astype('uint8')*255
p2_corrected=cv.morphologyEx(p2_corrected,cv.MORPH_OPEN,np.ones((3,3),np.uint8))
assert p2_residual.min()<0 # Negative differences remain negative instead of wrapping to 255.

### Work through resolution as a computation budget

Reducing width and height by two reduces pixel count by four, but small objects and thin gaps can disappear. Compare corrected full-resolution and half-resolution masks after nearest-neighbor resizing back to the original grid. Keep the same reference mask and report pixel IoU and object counts; do not call a lower pixel count an accuracy-preserving optimization without checking.

In [ ]:
p2_small=cv.resize(p2_scene,None,fx=.5,fy=.5,interpolation=cv.INTER_AREA).astype('float32')
p2_small_mask=((p2_small-cv.GaussianBlur(p2_small,(0,0),12.5))>25).astype('uint8')*255
p2_small_mask=cv.morphologyEx(p2_small_mask,cv.MORPH_OPEN,np.ones((3,3),np.uint8))
p2_small_up=cv.resize(p2_small_mask,(360,240),interpolation=cv.INTER_NEAREST)
p2_metrics=[]
for p2_name,p2_mask in [('global',p2_global),('normalized_full',p2_corrected),('normalized_half',p2_small_up)]:
    p2_binary=p2_mask>0;p2_true=p2_truth>0
    p2_iou=np.logical_and(p2_binary,p2_true).sum()/np.logical_or(p2_binary,p2_true).sum()
    p2_num,_,p2_stats,_=cv.connectedComponentsWithStats(p2_mask)
    p2_count=int((p2_stats[1:,cv.CC_STAT_AREA]>100).sum())
    p2_metrics.append({'method':p2_name,'iou':p2_iou,'objects_over_100_pixels':p2_count})
p2_report=pd.DataFrame(p2_metrics);display(p2_report)
assert p2_report.set_index('method').loc['normalized_full','iou']>p2_report.set_index('method').loc['global','iou']
p2_fig,p2_axes=plt.subplots(1,4,figsize=(12,3))
for p2_ax,p2_image,p2_title in zip(p2_axes,[p2_scene,p2_truth,p2_global,p2_corrected],['Observed','Reference','Global threshold','Normalized']):
    p2_ax.imshow(p2_image,cmap='gray');p2_ax.set_title(p2_title);p2_ax.axis('off')
plt.tight_layout();plt.savefig(p2_out/'opencv_illumination_report.png');plt.show()
p2_report.to_csv(p2_out/'opencv_mask_metrics.csv',index=False)

### Explain why the fix is conditional

The wide blur assumes illumination varies more slowly than object structure. A very large object can contaminate that estimate; this method is not a universal segmentation algorithm. The full and reduced resolutions trade detail for work. IoU is available because this fixture has ground truth; real deployment needs labeled images across actual lighting conditions.

**Try:** move two circles until they touch. **Worked reasoning:** connected components may merge them even with an accurate foreground mask. Threshold tuning cannot necessarily separate touching objects; consider distance transforms/watershed or a different instance-segmentation approach, then validate instance counts separately from foreground overlap.